## PASO 1 — Texto → Emoción

### Descripción de esta etapa

Vamos a trabajar en **dos fases** para construir el sistema de clasificación de emociones:

1. **Fase A (binaria):** `joy` vs `nojoy` (una emoción contra el resto).
2. **Fase B (multiclase):** varias emociones al mismo tiempo.

#### Fuentes de datos usadas kaggle

1. `data/Emotion Dataset for Emotion Recognition Tasks/training.csv`
   - 16,000 filas.
   - Columnas: `text` y `label` (codificada numéricamente).
   - sadness (0), joy (1), love (2), anger (3), fear (4).


2. `data/EmotionsDetectionDataset/emotion_dataset.csv`
   - 34,791 filas.
   - Columnas principales: `Text`, `Emotion`, `Clean_Text`
   - Emociones: `joy`, `sadness`, `fear`, `anger`, `surprise`, `neutral`, `disgust`, `shame`.

#### Cómo se unieron para la Fase A (joy vs nojoy)

Se construyó `data/joy_nojoy_combined.csv` con estas reglas:

- Si la etiqueta representa **joy**, asignamos `label_binary = 1` y `label_name = joy`.
- Cualquier otra emoción se asigna como `label_binary = 0` y `label_name = nojoy`.

Columnas del archivo combinado:

- `text`: texto de entrada.
- `label_binary`: 1 (`joy`) o 0 (`nojoy`).
- `label_name`: `joy` / `nojoy`.
- `source`: dataset de origen.

#### Para qué lo hacemos así

- Primero resolvemos un problema más simple (**1 emoción vs resto**).
- Luego escalamos a **varias emociones** con la misma base de trabajo.

# 1. Introducción

Clasificador de texto para distinguir entre **joy** y **nojoy**.

Objetivos de esta primera fase:

- Cargar y entender el dataset binario.
- Limpiar y preparar texto.
- Convertir texto en variables numéricas.
- Entrenar un modelo base.
- Evaluar desempeño y probar ejemplos manuales.

In [17]:
# Configuracion inicial e imports
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
 )

SEED = 42
np.random.seed(SEED)

print(f"Semilla fija: {SEED}")

Semilla fija: 42


# 2. Dataset

Cargamos el archivo combinado `joy_nojoy_combined.csv` que integra ambos datasets originales y contiene una etiqueta binaria:

- `label_binary = 1` para `joy`.
- `label_binary = 0` para `nojoy`.

In [18]:
# Cargar dataset binario joy vs nojoy
DATA_FILE = Path("data/joy_nojoy/joy_nojoy_combined.csv")

if not DATA_FILE.exists():
    raise FileNotFoundError(f"No se encontro el archivo: {DATA_FILE}")

df = pd.read_csv(DATA_FILE, usecols=["text", "label_binary", "source"]).copy()
df["label_binary"] = pd.to_numeric(df["label_binary"], errors="coerce")
df = df.dropna(subset=["text", "label_binary"]).reset_index(drop=True)
df["label_binary"] = df["label_binary"].astype(int)

valid_labels = {0, 1}
current_labels = set(df["label_binary"].unique())
if not current_labels.issubset(valid_labels):
    raise ValueError(f"Se encontraron etiquetas fuera de 0/1: {sorted(current_labels)}")

print("Primeras filas:")
display(df.head())

print("\nTamano del dataset:", df.shape)
print("\nDistribucion de clases (conteo):")
print(df["label_binary"].value_counts().sort_index())

print("\nDistribucion de clases (proporcion):")
print(df["label_binary"].value_counts(normalize=True).sort_index().round(3))

Primeras filas:


,text,label_binary,source
0,i didnt feel humiliated,0,training.csv
1,i can go from feeling so hopeless to so damned...,0,training.csv
2,im grabbing a minute to post i feel greedy wrong,0,training.csv
3,i am ever feeling nostalgic about the fireplac...,0,training.csv
4,i am feeling grouchy,0,training.csv



Tamano del dataset: (50792, 3)

Distribucion de clases (conteo):
label_binary
0    39747
1    11045
Name: count, dtype: int64

Distribucion de clases (proporcion):
label_binary
0    0.783
1    0.217
Name: proportion, dtype: float64


# 3. Preprocesamiento de texto

Aplicamos una limpieza básica para reducir ruido antes de vectorizar:

- Minúsculas
- Eliminación de URLs
- Eliminación de signos/puntuación
- Normalización de espacios
- Eliminación de filas vacías tras limpiar

In [19]:
# Limpieza de texto
URL_RE = re.compile(r"https?://\S+|www\.\S+")
NON_ALNUM_RE = re.compile(r"[^a-zA-Z0-9\s]")
MULTISPACE_RE = re.compile(r"\s+")

def clean_text(text):
    text = str(text).lower()
    text = URL_RE.sub(" ", text)
    text = NON_ALNUM_RE.sub(" ", text)
    text = MULTISPACE_RE.sub(" ", text).strip()
    return text

df["text"] = df["text"].fillna("")
df["clean_text"] = df["text"].map(clean_text)
df = df[df["clean_text"].str.len() > 0].reset_index(drop=True)

print("Tamano luego de limpieza:", df.shape)
display(df[["text", "clean_text", "label_binary"]].head())

Tamano luego de limpieza: (50792, 4)


,text,clean_text,label_binary
0,i didnt feel humiliated,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,im grabbing a minute to post i feel greedy wrong,0
3,i am ever feeling nostalgic about the fireplac...,i am ever feeling nostalgic about the fireplac...,0
4,i am feeling grouchy,i am feeling grouchy,0


# 4. Vectorización

Transformamos el texto limpio a una representación numérica con TF-IDF y luego hacemos split estratificado.


En esta sección usamos **TF-IDF** para convertir texto en números. El modelo no puede aprender directamente de frases, así que necesita una representación numérica de cada documento.

TF-IDF asigna más peso a palabras informativas y menos peso a palabras muy comunes. Después del split (`train/test`), ajustamos el vectorizador con `X_train` (`fit_transform`) y luego transformamos `X_test` (`transform`) para evitar fuga de información.

El resultado es una matriz numérica (`X_train_vec`, `X_test_vec`) que se usa como entrada para entrenar la regresión logística.

In [20]:
# Split y vectorizacion TF-IDF
X = df["clean_text"]
y = df["label_binary"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
 )

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=50000,
    min_df=2,
 )

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("X_train_vec:", X_train_vec.shape)
print("X_test_vec: ", X_test_vec.shape)

X_train_vec: (40633, 50000)
X_test_vec:  (10159, 50000)


# 5. Entrenamiento del modelo

Regresión Logística


# 6. Evaluación

Evaluamos con métricas de clasificación y matriz de confusión.

# 7. Pruebas